# ML v1.0: базовая модель на профильном датасете

Этот блокнот фиксирует первый baseline-эксперимент для файла `client_profile_v1_0_shuffled.csv`. Здесь мы подключаем Google Drive, загружаем версию `v1.0`, исключаем явные утечки и служебные поля, делаем базовый preprocessing и считаем первые метрики для baseline-моделей.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Запуск не в Colab: пропускаю mount Google Drive.")

BASE_PATH = Path("/content/drive/MyDrive/ieee-db")
DATA_FILE = BASE_PATH / "client_profile_v1_0_shuffled.csv"

print("Использую файл:", DATA_FILE)
if not DATA_FILE.exists():
    raise FileNotFoundError(f"Файл не найден: {DATA_FILE}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Использую файл: /content/drive/MyDrive/ieee-db/client_profile_v1_0_shuffled.csv


In [2]:
df = pd.read_csv(DATA_FILE)
print("Размер таблицы:", df.shape)
print("Распределение target:")
print(df["profile_fraud_label"].value_counts(dropna=False))
print()
print("Первые колонки:")
print(df.columns[:40].tolist())
display(df.head())

Размер таблицы: (174280, 95)
Распределение target:
profile_fraud_label
0    167785
1      6495
Name: count, dtype: int64

Первые колонки:
['client_id', 'tx_count_total', 'tx_amount_sum', 'tx_amount_mean', 'tx_amount_median_proxy', 'tx_amount_std', 'tx_count_standart_flow', 'tx_count_high_risk_flow', 'tx_sum_stanadart_flow_proxy_proxy', 'tx_sum_high_risk_flow_proxy', 'tx_freq_per_day', 'daily_activity_share', 'avg_inter_tx_seconds', 'short_turnover_share', 'amount_repeat_share', 'odd_amount_share', 'weighted_amount_repeat_share', 'cash_out_ratio_proxy', 'MCC_risk_share_proxy', 'high_risk_vs_mean', 'crypto_pattern_score', 'low_history_flag', 'history_support_score', 'productcd_nunique', 'addr2_nunique', 'card4_mode', 'card6_mode', 'tx_dt_span_days', 'top_email_domain', 'profile_fraud_label', 'identity_present', 'num_missing_identity', 'identity_rows', 'non_null_id_values', 'device_type_nunique', 'id_01_mean', 'id_01_median', 'id_01_count', 'id_02_mean', 'id_02_median']


,client_id,tx_count_total,tx_amount_sum,tx_amount_mean,tx_amount_median_proxy,tx_amount_std,tx_count_standart_flow,tx_count_high_risk_flow,tx_sum_stanadart_flow_proxy_proxy,tx_sum_high_risk_flow_proxy,...,id_29_mode,id_30_mode,id_31_mode,id_32_mode,id_33_mode,id_34_mode,id_35_mode,id_36_mode,id_37_mode,id_38_mode
0,12221_170.0_173,1,47.950,47.950000,47.950,0.000000,1,0,47.95,0.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5347__14,1,31.191,31.191000,31.191,0.000000,0,1,0.00,31.191,...,Found,NaN,chrome 62.0,NaN,NaN,NaN,F,F,T,T
2,7508_420.0_97,3,117.850,39.283333,36.950,7.133645,3,0,117.85,0.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,17759_126.0_12,1,103.950,103.950000,103.950,0.000000,1,0,103.95,0.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,16092_143.0_74,3,3426.550,1142.183333,1325.880,259.786317,3,0,3426.55,0.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Правила отбора признаков для v1.0

В версии `v1.0` мы используем постановку profile-level classification: одна строка соответствует одному клиенту, а целевая переменная показывает, был ли клиент fraud-like по полной доступной истории. Для первого baseline мы исключаем идентификатор `client_id`, любые явные утечки и очень разреженные поля с экстремальным количеством пропусков. Это не финальный production-пайплайн, а первая воспроизводимая точка отсчета, от которой дальше можно улучшать витрину и модель.

In [3]:
target_col = "profile_fraud_label"
id_cols = ["client_id"]
leaky_cols = [c for c in ["tx_count_fraud", "fraud_rate"] if c in df.columns]
high_missing_threshold = 0.95

high_missing_cols = [
    c for c in df.columns
    if c not in [target_col] + id_cols and df[c].isna().mean() > high_missing_threshold
]

feature_drop_cols = id_cols + leaky_cols + high_missing_cols

X = df.drop(columns=[c for c in feature_drop_cols + [target_col] if c in df.columns]).copy()
y = df[target_col].astype(int).copy()

numeric_cols = X.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

print("Удаляем служебные/разреженные колонки:", feature_drop_cols)
print("Количество признаков после отбора:", X.shape[1])
print("Числовых признаков:", len(numeric_cols))
print("Категориальных признаков:", len(categorical_cols))
print("Колонки с >95% missing:", high_missing_cols)

Удаляем служебные/разреженные колонки: ['client_id', 'id_07_mean', 'id_07_median', 'id_08_mean', 'id_08_median', 'id_21_mode', 'id_22_mode', 'id_23_mode', 'id_24_mode', 'id_25_mode', 'id_26_mode', 'id_27_mode']
Количество признаков после отбора: 82
Числовых признаков: 66
Категориальных признаков: 16
Колонки с >95% missing: ['id_07_mean', 'id_07_median', 'id_08_mean', 'id_08_median', 'id_21_mode', 'id_22_mode', 'id_23_mode', 'id_24_mode', 'id_25_mode', 'id_26_mode', 'id_27_mode']


In [4]:
from sklearn.model_selection import train_test_split

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.17647058823529413, random_state=42, stratify=y_train_full
)

print("Train shape:", X_train.shape, "Доля fraud:", round(y_train.mean(), 6))
print("Val shape:", X_val.shape, "Доля fraud:", round(y_val.mean(), 6))
print("Test shape:", X_test.shape, "Доля fraud:", round(y_test.mean(), 6))

Train shape: (121996, 82) Доля fraud: 0.037272
Val shape: (26142, 82) Доля fraud: 0.037258
Test shape: (26142, 82) Доля fraud: 0.037258


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False)),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="MISSING")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20)),
])

def make_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_cols),
            ("cat", categorical_transformer, categorical_cols),
        ],
        remainder="drop",
    )

def calc_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
        "precision_0_5": precision_score(y_true, y_pred, zero_division=0),
        "recall_0_5": recall_score(y_true, y_pred, zero_division=0),
        "f1_0_5": f1_score(y_true, y_pred, zero_division=0),
        "pred_positive_rate": float(y_pred.mean()),
    }

models = {
    "logreg": Pipeline(steps=[
        ("preprocessor", make_preprocessor()),
        ("model", LogisticRegression(max_iter=2000, solver="saga", class_weight="balanced", n_jobs=-1)),
    ]),
    "random_forest": Pipeline(steps=[
        ("preprocessor", make_preprocessor()),
        ("model", RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight="balanced_subsample", n_jobs=-1, random_state=42)),
    ]),
} 

In [6]:
results = []
fitted_models = {}

for name, model in models.items():
    print(f"Обучаю модель: {name}")
    model.fit(X_train, y_train)
    fitted_models[name] = model

    for split_name, X_part, y_part in [("val", X_val, y_val), ("test", X_test, y_test)]:
        y_prob = model.predict_proba(X_part)[:, 1]
        row = calc_metrics(y_part, y_prob)
        row["model"] = name
        row["split"] = split_name
        results.append(row)

results_df = pd.DataFrame(results)
results_df = results_df[["model", "split", "roc_auc", "pr_auc", "precision_0_5", "recall_0_5", "f1_0_5", "pred_positive_rate"]]
display(results_df.sort_values(["split", "pr_auc"], ascending=[True, False]))

results_path = BASE_PATH / "ml_v1_0_results.csv"
results_df.to_csv(results_path, index=False)
print("Таблица метрик сохранена в:", results_path)

best_model_name = (
    results_df[results_df["split"] == "val"]
    .sort_values("pr_auc", ascending=False)
    .iloc[0]["model"]
)

print("Лучшая модель по val PR-AUC:", best_model_name)
best_model = fitted_models[best_model_name]
test_prob = best_model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= 0.5).astype(int)
print(classification_report(y_test, test_pred, digits=4))

Обучаю модель: logreg


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Обучаю модель: random_forest


,model,split,roc_auc,pr_auc,precision_0_5,recall_0_5,f1_0_5,pred_positive_rate
3,random_forest,test,0.881020,0.420962,0.366503,0.536961,0.435652,0.054586
1,logreg,test,0.855962,0.294501,0.135748,0.731006,0.228976,0.200635
2,random_forest,val,0.883152,0.412430,0.343475,0.521561,0.414187,0.056576
0,logreg,val,0.850453,0.298371,0.134058,0.721766,0.226118,0.200597


Таблица метрик сохранена в: /content/drive/MyDrive/ieee-db/ml_v1_0_results.csv
Лучшая модель по val PR-AUC: random_forest
              precision    recall  f1-score   support

           0     0.9818    0.9641    0.9728     25168
           1     0.3665    0.5370    0.4357       974

    accuracy                         0.9482     26142
   macro avg     0.6741    0.7505    0.7042     26142
weighted avg     0.9588    0.9482    0.9528     26142

